# 2026-10-05 Lab data 3: the runs

**Author:** Kyle Shannon · **Notebook:** lab-data-3-the-runs · **Created:** 2026-10-06T08:15:28-04:00

**Purpose:** What the lab's run labels contain, which of their rows and times to trust, what is wrong with the event list and the rules that clean it, and what we deliberately leave out (the reward rows).

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = "retina"

import subprocess
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scienceplots  # noqa: F401  registers the SciencePlots styles

import axis_of_travel_rnn
from environments import list_environments, load_environment
from axis_of_travel_rnn.store import resolve_store

plt.style.use(["science", "notebook", "no-latex"])
rng = np.random.default_rng(0)
store = resolve_store(None)
FIGURES = Path("figures")

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
                        text=True).stdout.strip() or "unknown"
print(f"axis-of-travel-rnn {axis_of_travel_rnn.__version__}  commit {commit}  {date.today()}")
print(f"store: {store.root} ({store.source})")
print(f"environments: {', '.join(list_environments())}")

## Data

The lab's MATLAB files, read only, from the `LAB_DATA` folder in `configs/local.yaml`. Nothing is copied into the repository or the artifact store. The cells load them through `src/proj_helpers/lab_data_access.py` (the reader, the catalog and two ways to pick sessions) and nothing else. Every definition that encodes something we learned lives in the chapter that teaches it.

The plots are made from the lab's recordings, so run `pixi run nbstripout-install` once per clone: git then stores this notebook without its outputs.

`RUN_ALL = False` makes the cells that loop over sessions run on a sample of 12, the first three of each group. Set it to True for the full pass over all 109 sessions, which takes minutes on the network volume. The totals quoted in the text are for the full pass. Sessions named in the text always run.

In [ ]:
from collections import Counter, defaultdict

from proj_helpers.lab_data_access import catalog, find, lab_root, load_mat, pick_scope

RUN_ALL = False
np.set_printoptions(precision=3, suppress=True, linewidth=140)
root = lab_root()
sessions = catalog(root)
scope = pick_scope(sessions, RUN_ALL)
print(f"lab data: {root}")
print(f"{len(sessions)} sessions; the loops below cover {len(scope)}")

## Analysis

### 3.1 · What the labeler's event list holds

#### Why this was examined
The run table is the heart of the output: every statistic is "per run, per path". The first spec said runs come from
the lab's markers (`runTimeMarkers` and `runPosMarkers`, per path, start and end of every run) and that `clean` is the
sign of the run's label in `spiral_events`, "copied". That was written from variable names and shapes alone. When the
markers were first compared with the events in one baseline session (V4 step 2) the comparison crashed on a size
mismatch, 12 against 16. The crash was my bug, but it exposed the real thing: the markers hold fewer runs than the
events. They cannot both be "every run".

#### The question and the goal
What does each run variable actually hold? Which is the primary record? What do the label codes mean, and what else
is in the event list besides runs? Are there events in the arena (the schema had a `crossing` run kind and a `cued`
phase for cued arena crossings)? Goal: decide which variable the run table is built from, and which are checks.

#### What we found (V4 steps 1 to 3)
- Orientation. As loaded, `runTimeMarkers` and `runPosMarkers` are (nPaths, 2) cells: one row per path, column 0
  the starts, column 1 the ends, each cell a vector with one entry per run. The dictionary's old "(2, 12)" is the raw
  HDF5 shape, reversed. A session has only the paths it used: JL1_rec23 has pathList [1 2 3 4 9 10], not 12.
- `spiral_events` is (n, 3): tracking row, time, label. A start label is plus or minus the path (negative means
  dirty). The matching end label is plus or minus (100 + path), for example -4 then -104, or 10 then 110.
- JL1_rec23: 117 start rows, 117 end rows, 6 indecision rows (label 0), no reward rows (777), no other label. Starts
  and ends alternate.
- `nRuns` [12 14 6 19 17 22] equals the number of CLEAN starts per path (positive labels). Clean plus dirty starts
  give [16 14 9 20 29 29]. So `nRuns`, and with it the markers, count only clean runs (90 of 117); the 27 dirty runs
  exist only in `spiral_events`.
- The marker cells, times and rows, equal the clean runs from the events exactly. `spiral_events` is the primary
  record and the markers are its clean subset.
- The sweep over all sessions: 19,604 start rows, 19,603 end rows, 929 indecision rows, 36 reward rows, no other
  label anywhere. In the 108 sessions where starts and ends alternate, the end label is always the start label
  plus 100 with the same sign, `nRuns` equals the clean start count per path, and the markers equal the clean
  events, with no exceptions. 14,177 runs are clean and 5,313 dirty.
- No run event lies in the arena segment in any session (0 in segment 1; 30,685 in segment 2; 7,532 in segment 3).
- Indecision rows: 903 of the 918 that fall in a run sit in a dirty run.

#### What it changed
- Runs are built from `spiral_events`, so the table includes the dirty runs. The markers and `nRuns` become fail
  checks on the clean subset.
- `crossing` runs and the `cued` phase are dropped: there is no source for them. Arena frames are `forage`.
- Indecision rows are not stored; the dirty flag already carries nearly all of what they say.
- Reward rows are looked at in chapter 3.4.

#### Where it lives
Spec: Behavior, the runs bullet. Plan: Checks, Fail, runs. Dictionary: 1.2 (`spiral_events`, `nRuns`, the markers),
the label code table and 1.4. Converter: `build_runs`, `check_against_lab` (task S9).

#### How to run it
Sections 1 to 5 use one session each. Section 6 loops, so it follows `RUN_ALL`.

#### 0. Reading a label
Column 3 of `spiral_events` is the label, and it carries the whole structure:
- 1 to 99: the START of a run on that path. Positive means the labeler judged the run clean, negative means dirty.
- 101 to 199: the END of a run, the start label plus 100 with the same sign (for example -4 then -104).
- 0: an indecision point, a hesitation the labeler marked.
- 777: a reward click (chapter 3.4).
`event_kind` names each event by its label. `DT` is the seconds per frame as the lab stored it (chapter 1.2).

In [ ]:
DT = 0.016667


def event_kind(labels):
    """Name each event by its label: start (1 to 99, negative is dirty), end (101 to 199), indecision (0), reward (777)."""
    return np.select([labels == 0, labels == 777, np.abs(labels) < 100, np.abs(labels) < 200],
                     ["indecision", "reward", "start", "end"], default="other")

#### 1. The run variables in one baseline session
JL1_rec23. Look at the shapes: the marker cells are (6, 2), six paths by start and end. Look at `nRuns` and
`pathList`: six numbers each, and `nRuns` sums to 90.

In [ ]:
sid = find(sessions, "JL1_rec23")
RUN_VARIABLES = ["pathList", "nRuns", "nPaths", "runTimeMarkers", "runPosMarkers", "spiral_events"]
a = load_mat(sessions[sid]["analysis"], variables=RUN_VARIABLES)
print(sid)
for name in RUN_VARIABLES:
    print(f"  {name:15s}", np.shape(a[name]))
print("  nRuns   ", a["nRuns"])
print("  pathList", a["pathList"])
for name in ("runTimeMarkers", "runPosMarkers"):
    print(f"  {name}: shapes of the cells, first 4 rows")
    for row in a[name][:4]:
        print("   ", [np.shape(c) for c in np.atleast_1d(row)])
print("  spiral_events, first 12 rows (tracking row, time, label):")
print(a["spiral_events"][:12])

#### 2. The kinds of event, and do starts and ends pair up the way the labels say?
Expect: 117 start, 117 end, 6 indecision, no reward, no other. Starts and ends alternate, and the end label is the
start label plus 100 with the same sign.

In [ ]:
ev = a["spiral_events"]
labels = ev[:, 2]
kind = event_kind(labels)
print("event kinds:", {str(k): int(n) for k, n in zip(*np.unique(kind, return_counts=True))})
mask = (kind == "start") | (kind == "end")
starts, ends = ev[mask][0::2], ev[mask][1::2]
print("starts and ends alternate:", bool(all(kind[mask][0::2] == "start") and all(kind[mask][1::2] == "end")),
      " odd number of start/end rows:", int(mask.sum() % 2))
print("end label = start label + 100 with the same sign:",
      bool(np.array_equal(ends[:, 2], starts[:, 2] + 100 * np.sign(starts[:, 2]))))

#### 3. What `nRuns` and the markers count
This is the finding that reorganised the run table. Three lines per path: the lab's `nRuns`, the clean starts, and
clean plus dirty starts. `nRuns` should equal the clean row, not the last row. Then, per path, whether the marker
cells equal the clean runs from the events: all True means the markers are exactly the clean subset.

In [ ]:
paths = np.atleast_1d(a["pathList"]).astype(int)
print("per path, in pathList order:", paths.tolist())
print("  nRuns                 ", a["nRuns"].astype(int).tolist())
print("  starts, clean only    ", [int((starts[:, 2] == p).sum()) for p in paths])
print("  starts, clean + dirty ", [int((np.abs(starts[:, 2]) == p).sum()) for p in paths])
same_times, same_rows = [], []
for i, p in enumerate(paths):
    pick = starts[:, 2] == p                            # the markers hold the clean runs only
    same_times.append(bool(np.allclose(a["runTimeMarkers"][i, 0], starts[pick, 1])
                           and np.allclose(a["runTimeMarkers"][i, 1], ends[pick, 1])))
    same_rows.append(bool(np.array_equal(a["runPosMarkers"][i, 0], starts[pick, 0])
                          and np.array_equal(a["runPosMarkers"][i, 1], ends[pick, 0])))
print("  time markers equal the clean events", same_times)
print("  row markers equal the clean events ", same_rows)

#### 4. Indecision and reward rows
An indecision row (label 0) is a hesitation the labeler marked. The first one in this session sits inside a dirty run
and appears in the file AFTER that run's end row: the list is in the order the labeler clicked, not in time order
(chapter 3.3 returns to this). Reward rows (777) do not occur in JL1_rec23; NS14_rec18 has 32 of them.

In [ ]:
i = np.flatnonzero(kind == "indecision")[0]
print("the first indecision event with the rows around it:")
print(ev[max(0, i - 3):i + 3])
sid18 = find(sessions, "NS14_rec18")
ev18 = load_mat(sessions[sid18]["analysis"], variables=["spiral_events"])["spiral_events"]
j = np.flatnonzero(ev18[:, 2] == 777)[0]
print(f"\n{sid18}: the first reward event with the rows around it:")
print(ev18[max(0, j - 3):j + 3])

#### 5. Where do the events fall: arena or track?
The stamps (chapter 2.1) give the segments. Count the events inside each. The finding is that the first column, the
arena, is zero in every session.

In [ ]:
stamps = np.atleast_1d(load_mat(sessions[sid]["analysis"], variables=["timeStamps"])["timeStamps"]).astype(int)
print("events inside each segment (arena, track):",
      [int(((ev[:, 0] >= x) & (ev[:, 0] <= y)).sum()) for x, y in stamps.reshape(-1, 2)], " of", len(ev))

#### 6. The sweep over sessions
The same facts across sessions. Full-pass totals: events start 19,604, end 19,603, indecision 929, reward 36 (in 5
sessions); no other label; clean runs 14,177 and dirty 5,313; indecision rows inside a clean run 15, inside a dirty
run 903; events inside segment 1 zero, segment 2 30,685, segment 3 7,532. There must be no line printed under
"flagged" except `NS16_rec6_20150821_001` under starts and ends not alternating.

In [ ]:
NEEDED = ["spiral_events", "nRuns", "pathList", "runTimeMarkers", "runPosMarkers", "timeStamps"]


def same(x, y):
    x = np.atleast_1d(x)
    return x.shape == y.shape and bool(np.allclose(x, y))


tally, flagged = Counter(), defaultdict(list)
for sid, s in scope.items():
    a = load_mat(s["analysis"], variables=NEEDED)
    ev = np.asarray(a["spiral_events"], dtype=float).reshape(-1, 3)
    labels = ev[:, 2]
    start = (np.abs(labels) >= 1) & (np.abs(labels) < 100)
    end = (np.abs(labels) > 100) & (np.abs(labels) < 200)
    for name, m in (("start", start), ("end", end), ("indecision", labels == 0), ("reward 777", labels == 777)):
        tally[f"events: {name}"] += int(m.sum())
    tally["events: any other label"] += int((~(start | end | (labels == 0) | (labels == 777))).sum())

    se = ev[start | end]
    st, en = se[0::2], se[1::2]
    if len(se) % 2 or not ((np.abs(se[0::2, 2]) < 100).all() and (np.abs(se[1::2, 2]) > 100).all()):
        tally["sessions where starts and ends do not alternate"] += 1
        flagged["starts and ends do not alternate"].append(sid)
        continue
    tally["runs, clean"] += int((st[:, 2] > 0).sum())
    tally["runs, dirty"] += int((st[:, 2] < 0).sum())
    if not np.array_equal(en[:, 2], st[:, 2] + 100 * np.sign(st[:, 2])):
        flagged["end label is not start + 100"].append(sid)

    paths = np.atleast_1d(a["pathList"]).astype(int)
    if [int((st[:, 2] == p).sum()) for p in paths] != np.atleast_1d(a["nRuns"]).astype(int).tolist():
        flagged["nRuns is not the clean start count"].append(sid)
    cells_t = np.asarray(a["runTimeMarkers"], dtype=object).reshape(len(paths), 2)
    cells_r = np.asarray(a["runPosMarkers"], dtype=object).reshape(len(paths), 2)
    for i, p in enumerate(paths):
        pick = st[:, 2] == p
        if not (same(cells_t[i, 0], st[pick, 1]) and same(cells_t[i, 1], en[pick, 1])
                and same(cells_r[i, 0], st[pick, 0]) and same(cells_r[i, 1], en[pick, 0])):
            flagged["markers differ from the clean events"].append(sid)
            break

    for r in ev[labels == 0, 0]:
        hit = (st[:, 0] <= r) & (r <= en[:, 0])
        tally["indecision rows outside every run" if not hit.any() else
              "indecision rows inside a clean run" if st[hit][0, 2] > 0 else "indecision rows inside a dirty run"] += 1

    if "timeStamps" in a:
        segs = np.atleast_1d(a["timeStamps"]).astype(int).reshape(-1, 2)
        for k, (x, y) in enumerate(segs, 1):
            tally[f"events inside segment {k}"] += int(((ev[:, 0] >= x) & (ev[:, 0] <= y)).sum())

print(f"{len(scope)} sessions")
for key, n in sorted(tally.items()):
    print(f"  {n:6d}  {key}")
for key, sids in flagged.items():
    print(f"flagged: {key}: {len(sids)} sessions", sids[:10])

#### 7. (New, for looking) One session's runs along its timeline
Each bar is a run, drawn at its path number: black clean, red dirty. Dirty runs are the 27% the markers and `nRuns`
never mention. Gaps between bars are the dwell at the goal and the return to the start (what the spec calls
`between`).

In [ ]:
sid = find(sessions, "JL1_rec23")
ev = np.asarray(load_mat(sessions[sid]["analysis"], variables=["spiral_events"])["spiral_events"], dtype=float).reshape(-1, 3)
kind = event_kind(ev[:, 2])
se = ev[(kind == "start") | (kind == "end")]
st, en = se[0::2], se[1::2]
minutes = DT / 60
fig, ax = plt.subplots(figsize=(11, 3.5))
for (r0, _, label), (r1, _, _) in zip(st, en):
    ax.broken_barh([(r0 * minutes, (r1 - r0) * minutes)], (abs(label) - 0.4, 0.8),
                   facecolors="k" if label > 0 else "tab:red")
ax.set_xlabel("minutes into the recording")
ax.set_ylabel("path")
ax.set_title(f"{sid}: clean (black) and dirty (red) runs")
plt.show()

#### Things to try
- Do dirty runs differ in duration or length from clean runs on the same path? What does "dirty" select?
- Which paths does each rat use, and how does that change across days (the `pathList` per session)?
- Path sequence: what follows what? Count transitions between consecutive runs' path ids. This is the statistic a
  later behavioural generator would need, and it depends on the order rule in chapter 3.3.

### 3.2 · Rows, not times: which column of the event list is right?

#### Why this was examined
Each event in `spiral_events` stores a tracking row and a time for the same instant. In JL1_rec23 they agree
exactly: time = row * 0.016667, which is also how the tracking file's own time column is built (chapter 1.2). The
first all-session sweep (chapter 3.1, section 6) reported one number that made no sense: the largest
|time - row * 0.016667| over all events was 3,375 seconds, nearly an hour. If an event's row and time disagree, one
of them is wrong, and every run boundary and duration in the run table depends on which one we trust. The lab's
markers carry the same pair, so they were no help. Task V4, steps 4, 6 and 7.

#### The question and the goal
How widespread is the disagreement, which column is wrong, and what is the independent evidence? Goal: choose the
column the converter uses for run boundaries and justify it with something the lab did not derive from the same
columns.

#### What we found (V4 steps 4, 5, 6 and 7)
- The time column is not row * 0.016667 in 33 of 109 sessions (JL1 2, NS14 19, NS15 7, NS16 5). Within a session
  the offset is a constant: JL1_rec35 0.5 s, JL1_rec36 0.3 s, NS14_rec15 3175.9 s, NS14_rec17 3375.5 s, NS14_rec18
  611.9 s, NS14_rec19 753.5 s. Some events in the same session agree exactly, others carry the constant, and the
  shifted ones are spread through the file, never one block.
- In the two JL1 sessions only the START events are shifted (116 of 116, 120 of 120); the ends agree exactly. In NS
  both ends of a shifted run are shifted (3,704 shifted starts and 3,704 shifted ends across NS).
- Rows are valid tracking indices in every session: none lies past the end of the tracking.
- In every NS session at least 47 shifted events have a time that points past the end of the recording, so the
  time cannot be right there.
- JL1 is the hard case, because 0.3 and 0.5 s stay inside the recording. A position test (the front light at run
  starts clusters a little tighter by time, 6.3 and 8.5 px, than by row, 7.8 and 11.5 px) leaned toward the time,
  weakly, on two sessions.
- The deciding evidence is the lab's painted run label per frame (`pixelDvt`, last column). In four sessions
  (JL1_rec35, JL1_rec36, NS14_rec15, NS14_rec18) the first and last painted frame of every clean run equals the
  event's ROW: median offset 0.0, 100% within 2 frames, for shifted and unshifted events, starts and ends. Against
  the time the JL1 starts are off by -30 and -20 frames and the NS14 events by -190,551 and -36,714 frames.

#### What it changed
- Rows are authoritative. Every time in the tables is read from the frames at those rows; the event time column is
  never used.
- "Row and time markers disagree" cannot be a fail check (it would fail 33 sessions). It is a warn that counts the
  events whose time is not row * dt, with the offset.
- Caveat for the dictionary. In JL1_rec35 and JL1_rec36 the start rows may sit 0.3 to 0.5 s before the animal's own
  departure; the lab defined its runs by row, and that is the definition we copy.
- The painted-versus-rows check stays a warn until the first full converter run shows how clean it is everywhere.

#### Where it lives
Spec: Overview (last sentences) and Behavior, the runs bullet. Plan: Checks, the warn about event times and the
painted labels. Dictionary: 1.4 (Event time), 2.3 (`t_start`). Converter: `build_runs`, `check_painted` (task S9).

#### How to run it
The four sessions that carry the argument are always run (KEY). The list of all affected sessions in the last cell
follows `RUN_ALL`.

#### 0. Two definitions
- `read_events`: the labeler's click record, one row per event, (tracking row, time, label); the event list is described in chapter 3.1.
- `lights`: the two light positions with NaN where the unfilled FLAG is set (chapter 1.2). `DT` and `event_kind` are from chapter 3.1.

In [ ]:
def read_events(path):
    """The labeler's click record: one row per event, (tracking row, time, label)."""
    return np.asarray(load_mat(path, ["spiral_events"])["spiral_events"], dtype=float).reshape(-1, 3)


def lights(proc, unfilled):
    """The front and back light positions in camera pixels, NaN wherever that light is unfilled."""
    front, back = proc[:, 2:4].astype(float), proc[:, 4:6].astype(float)   # astype makes copies
    front[unfilled[:, 0]] = np.nan
    back[unfilled[:, 1]] = np.nan
    return front, back

In [ ]:
KEY = ["JL1_rec35", "JL1_rec36", "NS14_rec15", "NS14_rec18"]
key_ids = [find(sessions, k) for k in KEY]

#### 1. Event time minus row * dt, by kind of event
A "shifted" event has a gap of more than 0.01 s. Expect JL1_rec35: 116 of 240 shifted, median offset 0.5 s, all of
them starts. JL1_rec36: 120 shifted, 0.3 s, all starts. The NS14 sessions: starts and ends in equal numbers, offsets
in the hundreds or thousands of seconds, and a handful of indecision rows.

In [ ]:
for sid in key_ids:
    ev = read_events(sessions[sid]["analysis"])
    row, time, label = ev[:, 0], ev[:, 1], ev[:, 2]
    gap = time - row * DT
    shifted = np.abs(gap) > 0.01
    kind = event_kind(label)
    print(f"{sid}: {int(shifted.sum())} of {len(ev)} events shifted, median offset {np.median(gap[shifted]):.1f} s")
    for k in ("start", "end", "indecision", "reward"):
        m = kind == k
        print(f"    {k:10s} {int((m & shifted).sum()):4d} shifted of {int(m.sum()):4d}")

#### 2. (New, for looking) The offset down the file
Event index along the bottom, the gap between the event time and row * dt up the side. Two flat levels, zero and the
session's constant, and the shifted events interleaved with the good ones. In NS14_rec15 the shifted ones begin
partway through the file.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, key in zip(axes, ("JL1_rec35", "NS14_rec15")):
    sid = find(sessions, key)
    ev = read_events(sessions[sid]["analysis"])
    ax.plot(ev[:, 1] - ev[:, 0] * DT, ".", ms=3)
    ax.set_title(f"{sid[:-4]}")
    ax.set_xlabel("event index in the file")
    ax.set_ylabel("time - row * dt (s)")
plt.tight_layout()
plt.show()

#### 3. Are the rows valid, and do the shifted times point past the end?
Rows are valid if they lie between 1 and the number of tracking frames. A shifted time is impossible if the frame it
names, round(time / dt), lies beyond the end of the recording. Expect: no rows past the end anywhere; in NS14_rec15
all 108 shifted events point past the end, in NS14_rec18 89 of 210; in the JL1 sessions none do (their offsets are
small), which is why they needed the painted test.

In [ ]:
for sid in key_ids:
    s = sessions[sid]
    ev = read_events(s["analysis"])
    n = len(load_mat(s["dvt"], variables=["samplesUnfilled"])["samplesUnfilled"])
    row, time = ev[:, 0], ev[:, 1]
    shifted = np.abs(time - row * DT) > 0.01
    late = np.round(time[shifted] / DT) > n
    print(f"{sid:26s} {n:7d} frames; rows past the end: {int((row > n).sum())}; "
          f"{int(shifted.sum())} shifted events, {int(late.sum())} of them point past the end")

#### 4. The deciding test: the lab's painted frames
For every clean run, find the first and last frame the lab painted with that run's path number, near the run, and
compare with the event's row and with the frame its time implies. A difference of 0 means "the lab painted where
this column says". Expect rows: median 0.0 and 100% within 2 frames in every line. Times: 0% within 2 frames for the
shifted lines, with the JL1 starts at -30 and -20 and NS14 at -190,551 and -36,714.

In [ ]:
def edges(painted, p, a, b):
    """The first and last 1-based frame painted p between frames a and b, or None."""
    a = max(a, 1)
    hit = np.flatnonzero(painted[a - 1:b] == p)
    return (a + hit[0], a + hit[-1]) if len(hit) else None


for sid in key_ids:
    a = load_mat(sessions[sid]["analysis"], variables=["spiral_events", "pixelDvt"])
    ev = np.asarray(a["spiral_events"], dtype=float).reshape(-1, 3)
    painted = np.asarray(a["pixelDvt"])[:, -1]
    labels = ev[:, 2]
    se = ev[(np.abs(labels) >= 1) & (np.abs(labels) < 200) & (ev[:, 0] > 0)]
    st, en = se[0::2], se[1::2]
    clean = st[:, 2] > 0
    diffs, unpainted = {}, 0
    for first, last in zip(st[clean], en[clean]):
        found = edges(painted, int(first[2]), int(first[0]) - 120, int(last[0]) + 120)
        if found is None:
            unpainted += 1
            continue
        for name, frame, row, time in (("start", found[0], first[0], first[1]), ("end", found[1], last[0], last[1])):
            shifted = abs(time - row * DT) > 0.01
            diffs.setdefault((name, shifted), []).append((frame - row, frame - round(time / DT)))
    print(f"\n{sid}: painted edge minus the event's row, and minus its time, in frames (median, share within 2)")
    for (name, shifted), d in sorted(diffs.items()):
        d = np.array(d)
        print(f"  {name:5s} {'shifted  ' if shifted else 'unshifted'} n={len(d):4d}: "
              f"by row {np.median(d[:, 0]):7.1f} ({np.mean(np.abs(d[:, 0]) <= 2):4.0%}), "
              f"by time {np.median(d[:, 1]):9.1f} ({np.mean(np.abs(d[:, 1]) <= 2):4.0%})")
    print(f"  clean runs with no painted frame in their window: {unpainted}")

#### 5. The superseded position test, kept as a lesson
Before the painted test, the only evidence for JL1 was whether the animal is in a more consistent place at the
time-implied frame than at the row. For starts it is (a little). The painted record then showed the lab used the rows.
This is how a weak test misleads: two sessions, a modest improvement, and a plausible story (the start row is a few
frames early). The lab's definition wins because we copy the lab's segmentation, we do not detect our own.

In [ ]:
def groups_at(xy, frames, path, mask):
    at = xy[np.clip(np.round(frames).astype(int) - 1, 0, len(xy) - 1)]
    return [at[mask & (path == p)] for p in np.unique(path[mask])]


def spread(groups):
    d = [np.linalg.norm(g - np.nanmedian(g, axis=0), axis=1) for g in groups if len(g) >= 5]
    return float(np.nanmedian(np.concatenate(d))) if d else float("nan")


for key in ("JL1_rec35", "JL1_rec36"):
    sid = find(sessions, key)
    s = sessions[sid]
    ev = read_events(s["analysis"])
    d = load_mat(s["dvt"], variables=["processedDVT", "samplesUnfilled"])
    xy = lights(d["processedDVT"], d["samplesUnfilled"].astype(bool))[0]
    row, time, labels = ev[:, 0], ev[:, 1], ev[:, 2]
    bad = np.abs(time - row * DT) > 0.01
    path = np.abs(labels) % 100
    print(f"\n{sid}: median distance (px) of the front light from its path's median position at the event")
    for name, pick in (("start", (np.abs(labels) >= 1) & (np.abs(labels) < 100)),
                       ("end", (np.abs(labels) > 100) & (np.abs(labels) < 200))):
        print(f"  {name:5s} shifted ({int((pick & bad).sum())}): by row {spread(groups_at(xy, row, path, pick & bad)):6.1f}, "
              f"by time {spread(groups_at(xy, time / DT, path, pick & bad)):6.1f}; "
              f"unshifted ({int((pick & ~bad).sum())}) by row {spread(groups_at(xy, row, path, pick & ~bad)):6.1f}")

#### 6. Which sessions are affected?
Full pass: 33 sessions. The sample shows few or none; set RUN_ALL = True for the list. The bar chart sorts them by
offset: two JL1 sessions near zero, then NS14, NS15 and NS16 sessions from about 300 s upward.

In [ ]:
offsets = {}
for sid, s in scope.items():
    ev = read_events(s["analysis"])
    gap = ev[:, 1] - ev[:, 0] * DT
    bad = np.abs(gap) > 0.01
    if bad.any():
        offsets[sid] = (int(bad.sum()), len(ev), float(np.median(gap[bad])))
print(f"{len(offsets)} of {len(scope)} sessions have shifted event times")
for sid, (n_bad, n, off) in sorted(offsets.items(), key=lambda kv: kv[1][2]):
    print(f"  {sid:26s} {n_bad:4d} of {n:4d} events, offset {off:9.1f} s")
if offsets:
    fig, ax = plt.subplots(figsize=(10, 3))
    ordered = sorted(offsets.items(), key=lambda kv: kv[1][2])
    ax.bar(range(len(ordered)), [v[2] for _, v in ordered])
    ax.set_yscale("symlog")
    ax.set_ylabel("offset (s)")
    ax.set_xlabel("session, sorted by offset")
    plt.show()

#### Things to try
- Why would a labeling tool write a shifted time for only some events? The shifted starts in JL1 are 18 to 30
  frames late; in NS they are hundreds or thousands of seconds late. Are these two mechanisms or one?
- For a shifted NS event, is `time - offset` equal to a plausible tracking time? (It is: the session's constant
  removes the shift.) What clock could the shifted times be on?
- Recompute the run durations from the times in a JL1 session and from the rows. How different are they, as a
  fraction of a typical run?

### 3.3 · Cleaning the event list: click order, placeholders, duplicates, overlaps

#### Why this was examined
Once the event rows were trusted (chapter 3.2), the run table could be built from `spiral_events`. The first sweep
also printed "101 sessions out of time order or overlapping". My check was badly built: it tested the raw order of
the events and so mixed three different questions into one number. Pulling them apart (V4 steps 4 and 5) turned up
real defects in the list. They matter because a frame can belong to only one run, and because any later statistic
about which path follows which, or how long the pauses between runs are, depends on the runs being right and in
time order.

#### The question and the goal
What exactly is wrong with the event list, session by session, and can a short, explicit set of rules turn it into a
clean run list without disturbing the lab's own counts? Goal: the nine rules, each shown on the sessions that
motivated it, and a demonstration that the lab's `nRuns` and marker checks still hold.

#### What we found (V4 steps 4 and 5)
- Click order is the norm. 100 of 108 sessions are not in time order as stored (the file is mostly in time order,
  with the first break far in, at event 112 of 117 in JL1_rec23, and 157, 51, 115, 129 in others). The indecision
  row that appeared after its run's end in chapter 3.1 was the same thing.
- Row 0 placeholders. NS16_rec31_20151102_002 has a run with start row 0 (label -3) and end row 0 (label -103), time
  0: the one "reversed" run. NS16_rec6_20150821_001 has the same kind of pair (labels -2 and -102), and before it a
  dirty start (label -4, row 25,664) whose end does not follow it: 114 starts against 113 ends.
- 16 overlaps remain after sorting by start row. Eleven are boundary overlaps of 1 to 6 frames between neighbouring
  runs with different labels (JL1_rec35 1, JL1_rec36 8, NS15_rec15 2). Five are duplicates, the same label with
  nearly the same rows (NS14_rec23 2, NS14_rec29 1, NS14_rec6 2); the starts differ by 0 to 101 frames, the ends by
  1 to 3.
- `nRuns` passed on all 108 alternating sessions, so it counts the duplicates: the lab's checks have to be run on the
  raw events, before any deduplication.

#### What it changed: the nine rules (the converter's run-building)
1. Source is `spiral_events`, rows only. A warn counts events whose time is not row * dt.
2. Events at row 0 are dropped and counted.
3. Each start pairs with the next end in file order (end label = start label + 100, same sign). A start with no end
   is dropped and named in a warn.
4. Runs are sorted by start row, and `run` is numbered after sorting.
5. After sorting: same label and an overlap of at least half the shorter run is a DUPLICATE, the later one is
   dropped; different labels (or a small overlap) overlapping by up to 10 frames are CUT, the earlier run ending one
   frame before the next starts; any other overlap fails the session. None has been seen.
6. `clean` is the label's sign. Indecision rows are not stored; reward rows are ignored (chapter 3.4).
7. A run's epoch is the epoch holding its start row, blank when none does (chapter 2.3).
8. `nRuns` and the marker rows are fail checks, computed on the raw pairs before rules 2 to 5. The marker times and
   the painted column are warn checks.
9. `crossing` and `cued` are dropped (chapter 3.1).

#### Where it lives
Spec: Behavior, the runs bullet. Plan: Checks (Fail: runs; Warn: each rule's count). Dictionary: 1.4 (Faults in the
event list) and 2.3. Converter: `pair_runs`, `build_runs`, `check_against_lab` (task S9), each rule with its own test.

#### How to run it
The flagged sessions that motivate each rule are always run. The statistics over many sessions follow `RUN_ALL`.

#### 1. Click order
Pair the starts with the ends in file order and ask whether the start rows increase. Full pass: 100 of 108 sessions
do not. (The 109th, NS16_rec6, has an odd number of start and end rows and is set aside.) The example lines show
the first break: the run before it and the run after it.

In [ ]:
tally, examples = Counter(), []
for sid, s in scope.items():
    ev = read_events(s["analysis"])
    labels = ev[:, 2]
    se = ev[(np.abs(labels) >= 1) & (np.abs(labels) < 200)]
    if len(se) % 2:
        tally["odd number of start and end rows"] += 1
        continue
    tally["sessions paired"] += 1
    st, en = se[0::2], se[1::2]
    if (np.diff(st[:, 0]) < 0).any():
        tally["not in time order as stored"] += 1
        i = int(np.argmax(np.diff(st[:, 0]) < 0))
        examples.append((sid, i, st[i, [0, 2]].tolist(), float(en[i, 0]), st[i + 1, [0, 2]].tolist()))
print(dict(tally))
for sid, i, a, e, b in examples[:5]:
    print(f"  {sid:26s} run {i}: (row, label) {a} ends {e:.0f}; run {i + 1}: {b}")

#### 2. (New, for looking) The shape of click order
The start row of each run against its position in the file. A rising line with a few sudden drops back is a file
that was mostly written in time order and then had runs added or re-clicked out of place. Sorting by start row
straightens it.

In [ ]:
sid = find(sessions, "JL1_rec23")
ev = read_events(sessions[sid]["analysis"])
se = ev[(np.abs(ev[:, 2]) >= 1) & (np.abs(ev[:, 2]) < 200)]
st = se[0::2]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(st[:, 0], ".-", lw=0.5, label="as stored")
ax.plot(np.sort(st[:, 0]), "-", lw=1, label="sorted by start row")
ax.set_xlabel("run's position in the file")
ax.set_ylabel("start row")
ax.set_title(sid)
ax.legend()
plt.show()

#### 3. The two sessions with placeholder events
Expect NS16_rec31: a start and an end at row 0, time 0, labels -3 and -103. NS16_rec6: a start (-4, row 25,664) with
nothing after it but another start, a pair of row-0 events (-2, -102), then more normal events. The window shows the
events either side of the first place the start/end alternation breaks.

In [ ]:
for key in ("NS16_rec31", "NS16_rec6"):
    sid = find(sessions, key)
    ev = read_events(sessions[sid]["analysis"])
    se = ev[(np.abs(ev[:, 2]) >= 1) & (np.abs(ev[:, 2]) < 200)]
    starts = int((np.abs(se[:, 2]) < 100).sum())
    print(f"{sid}: {starts} starts, {len(se) - starts} ends; events at row 0: {int((se[:, 0] == 0).sum())}")
    is_start = np.abs(se[:, 2]) < 100
    wrong = np.flatnonzero(is_start != (np.arange(len(se)) % 2 == 0))
    if len(wrong):
        i = int(wrong[0])
        print("  alternation first breaks at position", i, "; rows around it (row, time, label):")
        print(se[max(0, i - 3):i + 3])

#### 4. The rules as code
`clean_runs` is rules 2 to 5 on one session's event array. It returns the kept runs as [start row, end row, label],
a Counter of what each rule did, and the raw pairs (before sorting and deduplication) for the lab check. The overlap
is measured in frames SHARED, previous end minus next start plus one; the validation quoted overlaps as previous end
minus next start, so a quoted "1 to 6 frames" is 2 to 7 shared frames. Both are well inside the limit of 10.

In [ ]:
OVERLAP_CUT_MAX = 10
COUNTS = ("events_at_row_0", "unmatched_events", "end_not_after_start", "duplicate_dropped", "overlap_cut", "overlap_left")


def pair_runs(events):
    """Pair each start with the end that follows it; returns (start row, end row, label) and the unmatched count."""
    pairs, pending, unmatched = [], None, 0
    for row, label in events:
        if abs(label) < 100:                                        # a start
            unmatched += int(pending is not None)                   # the previous start never ended
            pending = (row, label)
        elif pending is not None and label == pending[1] + 100 * np.sign(pending[1]):
            pairs.append((pending[0], row, pending[1]))
            pending = None
        else:                                                       # an end with no matching start
            unmatched += 1 + int(pending is not None)
            pending = None
    return pairs, unmatched + int(pending is not None)


def clean_runs(ev):
    rows, labels = ev[:, 0], ev[:, 2]
    is_run = ((np.abs(labels) >= 1) & (np.abs(labels) < 100)) | ((np.abs(labels) > 100) & (np.abs(labels) < 200))
    counts = Counter({name: 0 for name in COUNTS})
    counts["events_at_row_0"] = int((is_run & (rows == 0)).sum())
    keep = is_run & (rows > 0)
    pairs, counts["unmatched_events"] = pair_runs(list(zip(rows[keep], labels[keep])))
    runs = []
    for start, end, label in sorted(pairs, key=lambda p: p[0]):
        start, end = int(start), int(end)
        if end <= start:
            counts["end_not_after_start"] += 1
            continue
        if runs and runs[-1][1] >= start:
            shared = runs[-1][1] - start + 1
            shorter = min(runs[-1][1] - runs[-1][0] + 1, end - start + 1)
            if label == runs[-1][2] and shared >= shorter / 2:      # the same run labelled twice
                counts["duplicate_dropped"] += 1
                continue
            if shared <= OVERLAP_CUT_MAX:                           # neighbours that touch
                runs[-1][1] = start - 1
                counts["overlap_cut"] += 1
            else:
                counts["overlap_left"] += 1
        runs.append([start, end, label])
    return runs, counts, pairs


def lab_check(pairs, ana):
    """The lab's nRuns and runPosMarkers against the clean raw pairs: (nRuns agree, markers agree)."""
    paths = np.atleast_1d(ana["pathList"]).astype(int)
    clean = [p for p in pairs if p[2] > 0]
    counts = [sum(1 for p in clean if p[2] == path) for path in paths]
    n_ok = counts == np.atleast_1d(ana["nRuns"]).astype(int).tolist()
    cells = np.asarray(ana["runPosMarkers"], dtype=object).reshape(len(paths), 2)
    markers_ok = all(
        np.array_equal(np.sort([p[k] for p in clean if p[2] == path]), np.sort(np.atleast_1d(cells[i, k])))
        for i, path in enumerate(paths) for k in (0, 1))
    return n_ok, markers_ok

#### 5. The rules on the sessions that motivated them
What to expect, from the findings:
  NS16_rec31: 2 events at row 0 dropped; nothing else.
  NS16_rec6: 2 events at row 0, 1 unmatched event (the dirty -4 start), and the lab check still passes because the
             dropped start was dirty.
  NS14_rec23: 2 duplicates dropped. NS14_rec29: 1. NS14_rec6: 2.
  JL1_rec35: 1 overlap cut. JL1_rec36: 8. NS15_rec15: 2.
"overlap_left" and "end_not_after_start" must be 0 everywhere: those are the cases that would fail a session.

In [ ]:
flagged = ["NS16_rec31", "NS16_rec6", "NS14_rec23", "NS14_rec29", "NS14_rec6", "JL1_rec35", "JL1_rec36", "NS15_rec15"]
for key in flagged:
    sid = find(sessions, key)
    a = load_mat(sessions[sid]["analysis"], variables=["spiral_events", "nRuns", "pathList", "runPosMarkers"])
    ev = np.asarray(a["spiral_events"], dtype=float).reshape(-1, 3)
    runs, counts, pairs = clean_runs(ev)
    n_ok, markers_ok = lab_check(pairs, a)
    shown = {k: v for k, v in counts.items() if v}
    print(f"{sid:26s} {len(runs):4d} runs kept; {shown or 'nothing to fix'}; nRuns ok {n_ok}, markers ok {markers_ok}")

#### 6. The rules and the lab's checks, over many sessions
Full pass totals: events at row 0 4; unmatched events 1; duplicates dropped 5; overlaps cut 11; overlaps left 0; runs
ending before they start 0; and the lab check passing in every session, NS16_rec6 included.

In [ ]:
total, lab_fail = Counter(), []
for sid, s in scope.items():
    a = load_mat(s["analysis"], variables=["spiral_events", "nRuns", "pathList", "runPosMarkers"])
    ev = np.asarray(a["spiral_events"], dtype=float).reshape(-1, 3)
    runs, counts, pairs = clean_runs(ev)
    total.update(counts)
    total["runs kept"] += len(runs)
    n_ok, markers_ok = lab_check(pairs, a)
    if not (n_ok and markers_ok):
        lab_fail.append((sid, n_ok, markers_ok))
print({k: v for k, v in total.items()})
print("sessions where the lab's nRuns or markers disagree with the raw clean pairs:", lab_fail)

#### Things to try
- Change OVERLAP_CUT_MAX to 5 and rerun section 6. Which sessions now have "overlap_left"? That is how close the
  boundary overlaps are to the limit.
- For the five duplicates, print the two copies side by side (start, end, label). Do you agree the earlier start is
  the right one to keep? The largest start difference is 101 frames (NS14_rec6).
- Count transitions between consecutive runs' path ids with and without sorting. How different are they?
- What would a start with no end look like if it were a clean run? The rule drops it; the lab check would then fail
  the session. Is that the behaviour you want?

### 3.4 · Reward rows

#### Why this was examined
36 events carried the label 777, which the dictionary called "the reward location, the moment of arrival at the goal". The old schema had a `t_reward` column for it and a plan to use it to split the dwell at the goal from the return to the start. If the rows were a systematic marker, the run table should carry them. Task V4, step 8.

#### The question and the goal
Are the reward rows a systematic marker we can build `t_reward` and the dwell/return split from? Goal: decide, with evidence, what the converter does with them.

#### What we found
- Reward rows (777): 36 in 5 of 109 sessions. NS14_rec18 has 32, each 33 to 246 frames after the end of a run (the runs at even positions 0, 2, 4 ... 66 of the sorted list, paths 1 to 4, clean and dirty), each followed by 422 to 5,200 frames before the next run, in rows 21,386 to 65,737 only. NS14_rec22 has 1, 291 frames after the last run. JL1_rec32, NS15_rec10 and NS15_rec7 have 1 each, INSIDE a run (labels 9, 9, 4). So 777 is not a systematic marker.

#### What it changed
- `t_reward` is dropped from the run table. Reward rows are ignored, and the dictionary says why. The dwell/return split stays a stage 2 job without a reward anchor.

#### Where it lives
Dictionary: the label table (`777`), 1.4 and 2.3. Spec: Behavior, the runs bullet.

#### How to run it
The named sessions are always run. The scan in section 3 follows `RUN_ALL`.

#### 1. Reward rows
For each session that has any, place every reward row against the runs (the start and end rows, sorted by start).
Expect NS14_rec18 to show a regular pattern (a reward 33 to 246 frames after a run ends, with the next run 400 to
5,200 frames later) and the other four to show a single row each: one after the last run, three inside a run. A row
inside a run cannot be a reward time for that run in any consistent sense.

In [ ]:
reward_keys = ["JL1_rec32", "NS14_rec18", "NS14_rec22", "NS15_rec10", "NS15_rec7"]
after_end = {}
for key in reward_keys:
    sid = find(sessions, key)
    ev = read_events(sessions[sid]["analysis"])
    kind = event_kind(ev[:, 2])
    se = ev[((kind == "start") | (kind == "end")) & (ev[:, 0] > 0)]
    if len(se) % 2:
        print(f"{sid}: odd number of start and end rows, skipped")
        continue
    order = np.argsort(se[0::2, 0])
    starts, ends, lab = se[0::2, 0][order], se[1::2, 0][order], se[0::2, 2][order]
    places = Counter()
    for r in ev[kind == "reward", 0]:
        k = np.searchsorted(starts, r, side="right") - 1
        if k >= 0 and r <= ends[k]:
            places["inside a run"] += 1
            if len(ev[kind == "reward"]) <= 3:
                print(f"  {sid} row {r:.0f}: inside run {k} (label {lab[k]:.0f}) {starts[k]:.0f}-{ends[k]:.0f}")
        elif k >= 0:
            places["after a run's end"] += 1
            after_end.setdefault(sid, []).append(r - ends[k])
            if len(ev[kind == "reward"]) <= 3:
                print(f"  {sid} row {r:.0f}: {r - ends[k]:.0f} frames after run {k} ends")
        else:
            places["before the first run"] += 1
    print(f"{sid:26s} {int((kind == 'reward').sum()):3d} reward rows: {dict(places)}")

#### 2. (New, for looking) How regular is NS14_rec18's reward pattern?
The frames from the end of a run to the reward row that follows it. A tight cluster would look like a delivery
delay the labeler marked with a click; but it only happens in this one session, at one stretch of it, so it cannot
be a dataset-wide column.

In [ ]:
sid18 = find(sessions, "NS14_rec18")
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(after_end[sid18], bins=20)
ax.set_xlabel("frames from a run's end to the reward row")
ax.set_ylabel("rewards")
ax.set_title(f"{sid18[:-4]}: {len(after_end[sid18])} reward rows after a run")
plt.show()
print(f"min {min(after_end[sid18]):.0f}, median {np.median(after_end[sid18]):.0f}, max {max(after_end[sid18]):.0f} frames")

#### 3. How common are reward rows?
Full pass: 5 sessions of 109. Anything a converter builds on them would be empty for 104.

In [ ]:
counts = {}
for sid, s in scope.items():
    n = int((read_events(s["analysis"])[:, 2] == 777).sum())
    if n:
        counts[sid] = n
print(f"{len(counts)} of {len(scope)} sessions have reward rows:", counts)

#### Things to try
- If you wanted a dwell/return split without reward rows, what would you anchor it to? (Think of the run's end row and the next run's start row: the goal and start zones in the room frame, stage 2.)
- Are the NS14_rec18 reward rows at the same place on the track each time (the front light's position at the row)? If so, 777 may be a location marker used in that session only.

## Notes

Settled here:
- Runs are built from `spiral_events`; `nRuns` and the markers are its clean subset, so they are checks and not the source (chapter 3.1).
- Rows are authoritative: the event time column is shifted in 33 sessions, and the lab's painted frames match the rows (chapter 3.2).
- Nine rules turn the event list into a run table: drop row-0 placeholders, pair starts with ends, sort by start, drop duplicates, cut small overlaps, fail on any other overlap (chapter 3.3).
- Reward rows are ignored and `t_reward` is dropped (chapter 3.4).

Open: duplicates keep the earlier start, which is the longer or equal run in all five cases; and in two JL1 sessions the start rows may sit 0.3 to 0.5 s before the animal's own departure.

Next: write these three up as lab notes, then clean up the scratch files, then the converter.